# Notebook 01 — Excel Workbook Viewer

**Role in the pipeline:** first-pass *inspection* of the source workbook.
Reads one sheet (or all sheets), prints a quick console summary, and writes
a raw CSV plus a scrollable HTML table per sheet for offline browsing.

This notebook is **not** in the dependency chain for Stage 1A. The CSV that
feeds Stage 1A is written by Notebook 02 (TA CRM / pH-standard QC). Run this
notebook when you want to *look at* the workbook before processing it.

**How to run**
1. Edit the `Parameters` cell below (or override via Papermill).
2. `Kernel → Restart & Run All`.
3. Outputs land under `OUT_DIR` (or a `<workbook>__viewer_outputs/` folder
   beside the workbook if `OUT_DIR` is `None`).

See `01_excel_viewer.README.md` for the full design notes and the reasoning
behind each choice.


## Parameters

This is the only cell you should need to edit. Tag: `parameters` — Papermill
injects an `injected-parameters` cell immediately after this one when
overrides are supplied on the command line.


In [ ]:
# --- Parameters (override via papermill `-p NAME value`) -----------------

XLSX_PATH = r"C:\Users\OA_2023-03\OneDrive\Habitat Suitabilty model\OA\data_1\oa_prelim_data.xlsx"

# Output root. None -> a "<workbook_stem>__viewer_outputs/" folder is created
# next to the workbook. Set explicitly if you want it somewhere else.
OUT_DIR = None

# Which sheet(s) to read:
#   "0"   -> first sheet by index
#   "Foo" -> sheet named "Foo"
#   "all" -> every sheet in the workbook
SHEET = "0"

PREVIEW_ROWS = 15        # rows printed in the console quick-summary
HTML_MAX_ROWS = None     # rows shown in HTML table; None = all rows
OPEN_HTML = False        # open each HTML table in the default browser?
WRITE_RAW_CSV = True     # also write a raw CSV next to each HTML table?


## Setup

A single `%pip install` (the original notebook ran it twice). Imports kept
to what this notebook actually uses; everything reusable lives in
`oa_common.py`.


In [ ]:
%pip install --quiet openpyxl


In [ ]:
from __future__ import annotations

import sys
import webbrowser
from pathlib import Path

import pandas as pd

try:
    from IPython.display import display
except Exception:
    display = None

# Shared helpers (see oa_common.py for rationale)
from oa_common import (
    die,
    normalize_columns,
    print_quick_summary,
    read_excel_sheets,
    safe_sheet_name,
    utc_stamp,
    write_html_table,
    write_manifest,
)


## Resolve workbook path and load sheets

The workbook path is validated *before* any work begins. Failing fast with a
clear message (`die(...)`) is preferable to a midway crash; this is the
common "early validation" pattern recommended in the LA notebook
best-practices guide.


In [ ]:
xlsx_path = Path(XLSX_PATH).expanduser().resolve()
if not xlsx_path.exists():
    die(f"File not found: {xlsx_path}")
if xlsx_path.suffix.lower() != ".xlsx":
    die(f"Expected .xlsx, got: {xlsx_path.name}")

# Resolve SHEET parameter to something pandas understands.
if str(SHEET).lower() == "all":
    sheet_param: "str | int" = "all"
else:
    try:
        sheet_param = int(SHEET)
    except ValueError:
        sheet_param = SHEET

sheets = read_excel_sheets(xlsx_path, sheet_param)

# Output root. Note: we deliberately do NOT bake the workbook stem or any
# "stage" tag into output filenames -- the parent folder already carries
# that context. (Rationale: JWST pipeline file-naming convention; Palantir
# Foundry "descriptive names, no abbreviation accumulation".)
if OUT_DIR:
    out_root = Path(OUT_DIR).expanduser().resolve()
else:
    out_root = xlsx_path.parent / f"{xlsx_path.stem}__viewer_outputs"

out_root.mkdir(parents=True, exist_ok=True)

print(f"Workbook   : {xlsx_path}")
print(f"Sheets     : {list(sheets.keys())}")
print(f"Output root: {out_root}")


## Quick console preview of each sheet

Useful to confirm columns and row counts before writing anything to disk.


In [ ]:
for sheet_name, df in sheets.items():
    df = normalize_columns(df)
    print_quick_summary(df, sheet_name, preview_rows=PREVIEW_ROWS)
    print("\n" + "=" * 80 + "\n")


## Write outputs

Layout per run:

```
<out_root>/
    sheet_<safe_sheet>/
        data/raw.csv
        tables/table.html
    logs/manifest.json
```

The sheet's identity is in the folder name; the file role (`raw`, `table`)
is in the filename. No workbook stem, no stage tag — those would only
duplicate context that is already in the folder hierarchy.


In [ ]:
written_files: list[Path] = []

for sheet_name, df in sheets.items():
    df = normalize_columns(df)
    safe_sheet = safe_sheet_name(sheet_name)

    sheet_root = out_root / f"sheet_{safe_sheet}"
    data_dir = sheet_root / "data"
    tables_dir = sheet_root / "tables"
    data_dir.mkdir(parents=True, exist_ok=True)
    tables_dir.mkdir(parents=True, exist_ok=True)

    raw_csv = data_dir / "raw.csv"
    html_path = tables_dir / "table.html"

    if WRITE_RAW_CSV:
        df.to_csv(raw_csv, index=False)
        written_files.append(raw_csv)
        print(f"Wrote raw CSV : {raw_csv}")

    write_html_table(
        df,
        html_path,
        max_rows=HTML_MAX_ROWS,
        title=f"{xlsx_path.stem} — sheet {sheet_name}",
    )
    written_files.append(html_path)
    print(f"Wrote HTML    : {html_path}")

    if OPEN_HTML:
        webbrowser.open(html_path.as_uri())


## Summary of written files

In [ ]:
outputs_df = pd.DataFrame({"output_file": [str(p) for p in written_files]})
print(f"Total files written: {len(outputs_df)}")

if display is not None:
    display(outputs_df)
else:
    print(outputs_df.to_string(index=False))


## Manifest (provenance log)

The manifest is where input path, parameters, sheet list, output paths and
package versions are recorded. Keeping provenance here — instead of baking
it into every output filename — is what stops filename "stage-tag
accumulation" across the pipeline.


In [ ]:
logs_dir = out_root / "logs"
logs_dir.mkdir(parents=True, exist_ok=True)
manifest_path = logs_dir / "manifest.json"

manifest = {
    "notebook": "01_excel_viewer",
    "generated_utc": utc_stamp(),
    "input_xlsx": str(xlsx_path),
    "output_root": str(out_root),
    "parameters": {
        "XLSX_PATH": str(xlsx_path),
        "OUT_DIR": str(out_root),
        "SHEET": SHEET,
        "PREVIEW_ROWS": PREVIEW_ROWS,
        "HTML_MAX_ROWS": HTML_MAX_ROWS,
        "OPEN_HTML": OPEN_HTML,
        "WRITE_RAW_CSV": WRITE_RAW_CSV,
    },
    "sheets_read": list(sheets.keys()),
    "outputs": [str(p) for p in written_files],
    "package_versions": {
        "python": sys.version.split()[0],
        "pandas": pd.__version__,
    },
}

write_manifest(manifest_path, manifest)
print(f"Wrote manifest: {manifest_path}")


## Optional: preview the first exported CSV inline

In [ ]:
csv_files = [p for p in written_files if str(p).lower().endswith(".csv")]

if csv_files:
    first_csv = csv_files[0]
    print(f"Previewing: {first_csv}")
    preview_df = pd.read_csv(first_csv)
    if display is not None:
        display(preview_df.head(20))
    else:
        print(preview_df.head(20).to_string(index=False))
else:
    print("No CSV files were written (WRITE_RAW_CSV is False).")
